# Spidey pilot1 evaluation
Scores the new model (`spidey-pilot1.cact`) and, for comparison, your current `spidey.cact` on the same held-out rows.
Needs a Drive folder `MyDrive/spidey_eval/` with: `spidey-pilot1.cact`, `intents_old_test.jsonl`, `intents_new_test.jsonl`, `ui_test_300.jsonl`.
Everything is synthetic or your own intent data; no phone data. Telemetry is turned off.

In [ ]:
%%capture
!pip install -q cactus-needle huggingface_hub

In [ ]:
import os, json, time, collections
os.environ["NEEDLE_TELEMETRY"] = "0"
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/spidey_eval"
print(os.listdir(D))
import needle, importlib.metadata as m
print("cactus-needle", m.version("cactus-needle"))

In [ ]:
from huggingface_hub import hf_hub_download
BASE = hf_hub_download("abhinayambati/spidey", "spidey.cact")   # your current published model
NEW = f"{D}/spidey-pilot1.cact"
print(BASE, os.path.getsize(BASE), NEW, os.path.getsize(NEW))

In [ ]:
norm = lambda v: v.strip().strip(".!?").casefold() if isinstance(v, str) else v
canon = lambda calls, f: [(c["name"], tuple(sorted((k, f(v)) for k, v in c["arguments"].items()))) for c in calls]

def load(p): return [json.loads(l) for l in open(p, encoding="utf8") if l.strip()]

def score(weights, rows, tools_override=None, limit=None, tag=""):
    agents, stats, lat, out = {}, collections.defaultdict(collections.Counter), [], []
    for row in rows[:limit]:
        tools = tools_override or row["tools"]
        key = json.dumps(tools, sort_keys=True)
        if key not in agents:
            agents[key] = needle.Needle(tools=tools, weights=weights, auto_date=False)
        ag = agents[key]; ag.reset(); gold = row["answers"]
        kind = "negative" if not gold else (gold[0]["name"] if len(gold) == 1 else "multi")
        t0 = time.time()
        try: r = ag.complete(row["query"])
        except Exception as e: r = {"function_calls": [], "error": repr(e)}
        lat.append(time.time() - t0)
        pred = r.get("function_calls") or []
        ok = canon(pred, norm) == canon(gold, norm)
        nm = [c["name"] for c in pred] == [c["name"] for c in gold]
        for k in (kind, "ALL"):
            s = stats[k]; s["n"] += 1; s["ok"] += ok; s["nm"] += nm
        out.append({"query": row["query"], "ok": ok, "gold": gold, "pred": pred})
    print(f"== {tag} ({len(out)} rows)  mean {sum(lat)/len(lat):.2f}s/query")
    print(f"{'group':<16}{'n':>5}{'exact':>8}{'tool':>8}")
    for k in sorted(stats, key=lambda x: (x == 'ALL', x)):
        s = stats[k]; print(f"{k:<16}{s['n']:>5}{s['ok']/s['n']:>8.1%}{s['nm']/s['n']:>8.1%}")
    return out

## 1. Did the original five intents regress? (same rows, both models)
The current model only knows 5 tools, so it gets the 5-tool list; the new model gets the 10-tool list its rows carry.

In [ ]:
old = load(f"{D}/intents_old_test.jsonl")
tools5 = old[0]["tools"][:5]
base_old = score(BASE, old, tools_override=tools5, tag="CURRENT spidey.cact, original intents")
new_old  = score(NEW,  old, tag="PILOT1, original intents (10-tool list)")

## 2. New intents (pilot1 only; the current model does not know them)

In [ ]:
newi = load(f"{D}/intents_new_test.jsonl")
new_new = score(NEW, newi, tag="PILOT1, new intents")

## 3. Screen actions (synthetic held-out screens; 300-row sample; pilot1 only)
Set `LIMIT` lower if it is too slow on the free CPU.

In [ ]:
LIMIT = 300
ui = load(f"{D}/ui_test_300.jsonl")
new_ui = score(NEW, ui, limit=LIMIT, tag="PILOT1, screen actions")

In [ ]:
json.dump({"base_old": base_old, "new_old": new_old, "new_new": new_new, "new_ui": new_ui},
          open(f"{D}/results.json", "w"), ensure_ascii=False)
print("saved", f"{D}/results.json")